In [1]:
import pandas as pd
from collections import deque

# -----------------------------
# STEP 1: LOAD DATA
# -----------------------------
df = pd.read_excel("Masked_Twitter, Inc. (N.D. Cal.) (2016).xlsx", engine="openpyxl")


In [2]:
df.head()

,ISIN/Cusip,Fund Name,Fund Number,Transaction Type,Purchases,Sales,Holdings,Price per share,Trade Date,Currency,Entity,Comments
0,US90184L1026,Fund 5,Fund 5,Beginning Holdings,NaN,NaN,900.0,NaN,2015-02-05,USD,Client 1-1,NaN
1,US90184L1026,Fund 5,Fund 5,Purchase,100.0,NaN,NaN,50.64,2015-04-02,USD,Client 1-1,NaN
2,US90184L1026,Fund 5,Fund 5,Sale,NaN,100.0,NaN,37.50,2015-05-19,USD,Client 1-1,NaN
3,US90184L1026,Fund 5,Fund 5,Sale,NaN,900.0,NaN,36.46,2015-05-27,USD,Client 1-1,NaN
4,US90184L1026,Fund 5,Fund 5,End Holdings,NaN,NaN,0.0,NaN,2015-10-30,USD,Client 1-1,NaN


In [3]:
# -----------------------------
# STEP 2: PREPROCESS
# -----------------------------
df["Trade Date"] = pd.to_datetime(df["Trade Date"], dayfirst=True)

df["Purchases"] = df["Purchases"].fillna(0)
df["Sales"] = df["Sales"].fillna(0)

df["Qty"] = df["Purchases"] - df["Sales"]

df = df[df["Transaction Type"].isin(["Purchase", "Sale"])]

df = df.sort_values(["Entity", "Fund Name", "Trade Date"])

In [4]:
# -----------------------------
# STEP 3: TABLE 1 LOGIC
# -----------------------------

def get_purchase_bucket(date):
    if date < pd.Timestamp("2015-04-28"):
        return "early"
    elif date == pd.Timestamp("2015-04-28"):
        return "mid"
    elif date <= pd.Timestamp("2015-07-28"):
        return "late"
    else:
        return "post"


def get_sale_bucket(date):
    if date < pd.Timestamp("2015-04-28"):
        return "pre"
    elif date == pd.Timestamp("2015-04-28"):
        return "same_day"
    elif date <= pd.Timestamp("2015-07-28"):
        return "period1"
    elif date <= pd.Timestamp("2015-07-30"):
        return "period2"
    elif date == pd.Timestamp("2015-07-31"):
        return "period3"
    else:
        return "post_aug1"

In [5]:
inflation_table = {

    "early": {
        "pre": 0.00,
        "same_day": 8.97,
        "period1": 12.93,
        "period2": 18.27,
        "period3": 18.69,
        "post_aug1": 20.34
    },

    "mid": {
        "same_day": 0.00,
        "period1": 3.96,
        "period2": 9.30,
        "period3": 9.72,
        "post_aug1": 11.37
    },

    "late": {
        "period1": 0.00,
        "period2": 5.34,
        "period3": 5.76,
        "post_aug1": 7.41
    },

    "post": {
        "period2": 0.00,
        "period3": 0.00,
        "post_aug1": 0.00
    }
}

In [6]:
def get_inflation_decline(purchase_date, sale_date):
    p_bucket = get_purchase_bucket(purchase_date)
    s_bucket = get_sale_bucket(sale_date)

    return inflation_table.get(p_bucket, {}).get(s_bucket, 0.0)

In [7]:
# -----------------------------
# STEP 4: CONSTANTS
# -----------------------------
AVG_PRICE = 28.06  # From Table 2

In [8]:
# -----------------------------
# STEP 5: LOSS CALCULATION
# -----------------------------

def calculate_loss(buy, sell, qty):

    buy_price = buy["price"]
    sell_price = sell["Price per share"]

    buy_date = buy["date"]
    sell_date = sell["Trade Date"]

    inflation = get_inflation_decline(buy_date, sell_date)

    # Case A
    if sell_date < pd.Timestamp("2015-04-28"):
        return 0

    # Case B
    elif sell_date <= pd.Timestamp("2015-08-02"):
        loss = min(
            inflation,
            buy_price - sell_price
        )

    # Case C
    elif sell_date <= pd.Timestamp("2015-10-30"):
        loss = min(
            inflation,
            buy_price - sell_price,
            buy_price - AVG_PRICE
        )

    # Case D
    else:
        loss = min(
            inflation,
            buy_price - AVG_PRICE
        )

    return max(loss, 0) * qty

In [9]:
# -----------------------------
# STEP 6: FIFO MATCHING
# -----------------------------

def compute_loss_for_fund(df_fund):

    inventory = deque()
    total_loss = 0

    total_purchase = 0
    total_sales = 0

    for _, row in df_fund.iterrows():

        qty = row["Qty"]
        price = row["Price per share"]
        date = row["Trade Date"]

        if qty > 0:
            inventory.append({
                "qty": qty,
                "price": price,
                "date": date
            })
            total_purchase += qty * price

        elif qty < 0:
            sell_qty = abs(qty)
            total_sales += sell_qty * price

            while sell_qty > 0 and inventory:

                buy = inventory[0]
                matched_qty = min(sell_qty, buy["qty"])

                loss = calculate_loss(buy, row, matched_qty)
                total_loss += loss

                buy["qty"] -= matched_qty
                sell_qty -= matched_qty

                if buy["qty"] == 0:
                    inventory.popleft()
        

    # Remaining holdings
    holding_value = sum(
        item["qty"] * AVG_PRICE for item in inventory
    )

     # -----------------------------
    # STEP 7: MARKET LOSS CAP
    # -----------------------------
    market_loss = total_purchase - (total_sales + holding_value)

    if market_loss <= 0:
        return 0

    return min(total_loss, market_loss)

In [10]:
# -----------------------------
# STEP 8: APPLY PER CLIENT/FUND
# -----------------------------

results = []

for (client, fund), group in df.groupby(["Entity", "Fund Name"]):

    loss = compute_loss_for_fund(group)

    results.append({
        "Client": client,
        "Fund": fund,
        "Loss": loss
    })

result_df = pd.DataFrame(results)


In [11]:
# -----------------------------
# STEP 9: CLIENT LEVEL
# -----------------------------
client_loss = result_df.groupby("Client")["Loss"].sum().reset_index()

In [12]:

# -----------------------------
# STEP 10: OUTPUT
# -----------------------------
print("Fund-wise Loss:")
print(result_df)

print("\nClient-wise Loss:")
print(client_loss)

Fund-wise Loss:
        Client      Fund       Loss
0   Client 1-1    Fund 5      0.000
1   Client 1-2   Fund 21   2545.525
2   Client 1-2   Fund 22      0.000
3   Client 1-2   Fund 43      0.000
4   Client 1-2   Fund 44      0.000
5   Client 1-2   Fund 63      0.000
6   Client 1-2   Fund 64      0.000
7     Client 2  Fund 102      0.000
8     Client 2  Fund 103      0.000
9     Client 2  Fund 111      0.000
10    Client 2  Fund 112      0.000
11    Client 2  Fund 116      0.000
12    Client 2  Fund 123      0.000
13    Client 2  Fund 135      0.000
14    Client 2  Fund 138      0.000
15    Client 2  Fund 139      0.000
16    Client 2  Fund 158      0.000
17    Client 2   Fund 91      0.000
18    Client 2   Fund 95  22200.000

Client-wise Loss:
       Client       Loss
0  Client 1-1      0.000
1  Client 1-2   2545.525
2    Client 2  22200.000
